# The viscoelastic FWI gradient: theory and where it lives in SeisCL

SeisCL computes the gradient of the waveform misfit with respect to
$(\rho, M, \mu, \tau_p, \tau_s)$ by the adjoint-state method. The expressions it
uses are not ad hoc — they come from Chapter 3 of

> Fabien-Ouellet, G. (2017). *Inversion des formes d'ondes complètes
> viscoélastique.* PhD thesis, INRS.
> [PDF](https://espace.inrs.ca/id/eprint/5251/1/Fabien-Ouellet,%20Gabriel.pdf)

specifically §3.5 and equations (3.51)–(3.55).

This notebook is a map between that derivation and the source. It covers:

1. where the gradient expressions come from,
2. the five parameter gradients and the six dot products they are built from,
3. **exactly which line of `calc_grad.c` implements which term**,
4. a symbolic check that the coded coefficients equal the published ones,
5. the parts that are still open.

It runs on `sympy` and `numpy` alone — no SeisCL build or GPU needed.

## 1. Where the expressions come from

The velocity–stress–memory system is written (thesis eq. 3.20) as

$$\mathbf{A}\dot{\boldsymbol\phi} + \mathbf{B}\boldsymbol\phi - \mathbf{G}\mathbf{C}\boldsymbol\phi - \mathbf{s} = 0$$

with $\boldsymbol\phi = (\mathbf v, \boldsymbol\sigma, \mathbf R^1,\dots,\mathbf R^L)$.
$\mathbf G$ is block-diagonal in the material properties. The adjoint-state
gradient (eq. 3.36, specialised to isotropy in 3.51) is

$$\frac{\partial X}{\partial m_\alpha} = -\Big\langle \boldsymbol\psi,\; \mathbf T\frac{\partial \boldsymbol\Lambda^{-1}}{\partial m_\alpha}\mathbf T\big(\mathbf A\dot{\boldsymbol\phi} + \mathbf B\boldsymbol\phi - \mathbf s\big)\Big\rangle$$

where $\mathbf G = \mathbf T\boldsymbol\Lambda\mathbf T$ is an eigendecomposition
(eq. 3.44–3.50). **In isotropy the eigenvalues are the bulk and shear
combinations $3\lambda+2\mu$ and $2\mu$** (eq. 3.46/3.48).

That is worth remembering, because differentiating $\boldsymbol\Lambda^{-1}$ is
what produces the factor

$$\big(N_d M - 2(N_d-1)\mu\big)^{-2}$$

which you will meet again and again in `calc_grad.c` as `1/fact1` and
`1/fact2`. It is not a fudge factor; it is the inverse-squared eigenvalue.

## 2. The five gradients and the six dot products

Thesis eq. (3.52), for $m = [\rho, M, \mu, \tau_p, \tau_s]$:

$$\begin{aligned}
\frac{\partial X}{\partial \rho} &= \langle \tilde v_x, \partial_t v_x\rangle + \langle \tilde v_y, \partial_t v_y\rangle + \langle \tilde v_z, \partial_t v_z\rangle\\
\frac{\partial X}{\partial M}    &= -c_1^{M}P_1 + c_2^{M}P_2\\
\frac{\partial X}{\partial \tau_p} &= -c_1^{\tau_p}P_1 + c_2^{\tau_p}P_2\\
\frac{\partial X}{\partial \mu}  &= -c_1^{\mu}P_3 + c_2^{\mu}P_1 - c_3^{\mu}P_4 + c_4^{\mu}P_5 - c_5^{\mu}P_2 + c_6^{\mu}P_6\\
\frac{\partial X}{\partial \tau_s} &= -c_1^{\tau_s}P_3 + c_2^{\tau_s}P_1 - c_3^{\tau_s}P_4 + c_4^{\tau_s}P_5 - c_5^{\tau_s}P_2 + c_6^{\tau_s}P_6
\end{aligned}$$

The six dot products (eq. 3.53) split into **stress** terms and
**memory-variable** terms:

| | quantity | kind |
|---|---|---|
| $P_1$ | $\langle \tilde\sigma_{xx}+\tilde\sigma_{yy}+\tilde\sigma_{zz},\ \partial_t(\sigma_{xx}+\sigma_{yy}+\sigma_{zz})\rangle$ | stress, trace |
| $P_2$ | $\langle \tilde R_{xx}+\tilde R_{yy}+\tilde R_{zz},\ (1+\tau_\sigma^l\partial_t)(r_{xx}+r_{yy}+r_{zz})\rangle$ | memory, trace |
| $P_3$ | $\sum \langle \tilde\sigma_{ij}, \partial_t\sigma_{ij}\rangle$, $ij \in \{xy,xz,yz\}$ | stress, shear |
| $P_4$ | $\sum \langle \tilde\sigma_{ii}, \partial_t((N_d-1)\sigma_{ii}-\sigma_{jj}-\sigma_{kk})\rangle$ | stress, deviatoric |
| $P_5$ | $\sum \langle \tilde R_{ij}, (1+\tau_\sigma^l\partial_t)r_{ij}\rangle$ | memory, shear |
| $P_6$ | $\sum \langle \tilde R_{ii}, (1+\tau_\sigma^l\partial_t)((N_d-1)r_{ii}-r_{jj}-r_{kk})\rangle$ | memory, deviatoric |

Note $\rho$'s gradient involves **only** the velocity correlation — no stress,
no memory. That is why in SeisCL `gradrho` gets `-dot[8]` and nothing else,
and why the $M$/$\mu$ contributions to $\rho$ in a $(v_p,v_s,\rho)$
parameterization are a *separate* chain rule applied afterwards, not part of
the correlation.

## 3. Mapping onto the source

The correlation lives in `src/calc_grad.c` (host reference) and in the
`src/grad_dft*.cl` device kernels. The thesis $P_i$ map onto the code's
`dot[]` array like this:

| thesis | code | note |
|---|---|---|
| $P_1$ | `dot[0]`, and `dot[3]` | |
| $P_2$ | `dot[1]`, and `dot[6]` | |
| $P_3$ | `dot[2]` | |
| $P_4$ | `dot[4]` | |
| $P_5$ | `dot[5]` | |
| $P_6$ | `dot[7]` | |

`calc_grad.c` contains the lines `dot[3]=dot[0];` and `dot[6]=dot[1];`. Those
look like redundant copies but are not: eq. (3.52d–e) genuinely reuse $P_1$
and $P_2$ inside the $\mu$ and $\tau_s$ expressions, and the duplicate slots
keep the coefficient indices lined up with the published ordering.

The accumulation then reads, term for term:

```c
gradM[indm]    += -c[0]*dot[0] + c[1]*dot[1];
gradtaup[indm] += -c[8]*dot[0] + c[9]*dot[1];
gradmu[indm]   += -c[2]*dot[2] + c[3]*dot[3] - c[4]*dot[4]
                  +c[5]*dot[5] - c[6]*dot[6] + c[7]*dot[7];
gradtaus[indm] += -c[10]*dot[2]+ c[11]*dot[3]- c[12]*dot[4]
                  +c[13]*dot[5]- c[14]*dot[6]+ c[15]*dot[7];
gradrho[indm]  += -dot[8];
```

so `c[0..1]` are $c^M_{1,2}$, `c[2..7]` are $c^\mu_{1..6}$, `c[8..9]` are
$c^{\tau_p}_{1,2}$ and `c[10..15]` are $c^{\tau_s}_{1..6}$. The coefficients
themselves are built in `grad_coefvisc_1()` (and its `_SH` twin).

## 4. Checking the coded coefficients against the thesis

Eq. (3.54)–(3.55). Two conventions have to be reconciled first:

* the thesis's $\tau$ is the **total** relaxation level, SeisCL's is
  **per mechanism**, so $\tau_{\text{thesis}} = L\,\tau_{\text{code}}$ — which is
  why the thesis writes $(1+\tau_p)$ where the code writes `(1+L*taup)`, and
  why $\partial/\partial\tau_{\text{code}} = L\,\partial/\partial\tau_{\text{thesis}}$
  puts an extra $L$ on the four $\tau$ coefficients;
* `calc_grad.c` sets $\alpha = 0$ unconditionally (`al=0`, its only
  assignment), so every $(1+\alpha\tau)$ factor collapses to 1 in practice.

Let's verify symbolically rather than by eye.

In [1]:
import sympy as sp

M, mu, tp, ts, a, L, Nd = sp.symbols(
    'M mu tau_p tau_s alpha L N_d', positive=True)

# --- thesis eq. (3.55): the two inverse-squared eigenvalue factors ----------
b1 = (Nd*M*(1+L*tp)/(1+a*tp) - 2*(Nd-1)*mu*(1+L*ts)/(1+a*ts))**-2
b2 = (Nd*M*tp/(1+a*tp)       - 2*(Nd-1)*mu*ts/(1+a*ts))**-2

# --- thesis eq. (3.54) -----------------------------------------------------
thesis = {
 'c1M':  (1+L*tp)/(1+a*tp)*b1,           'c2M':  tp/(1+a*tp)*b2,
 'c1tp': (1-a)*M/(1+a*tp)**2*b1,         'c2tp': M/(1+a*tp)**2*b2,
 'c1mu': (1+a*ts)/(mu**2*(1+L*ts)),      'c1ts': (1-a)/(mu*(1+L*ts)**2),
 'c2mu': (Nd+1)/3*(1+L*ts)/(1+a*ts)*b1,  'c2ts': (Nd+1)/3*(1-a)*mu/(1+a*ts)**2*b1,
 'c3mu': (1+a*ts)/(2*Nd*mu**2*(1+L*ts)), 'c3ts': (1-a)/(2*Nd*mu*(1+L*ts)**2),
 'c4mu': (1+a*ts)/(mu**2*ts),            'c4ts': 1/(mu*ts**2),
 'c5mu': (Nd+1)/3*ts/(1+a*ts)*b2,        'c5ts': (Nd+1)/3*mu/(1+a*ts)**2*b2,
 'c6mu': (1+a*ts)/(2*Nd*mu**2*ts),       'c6ts': 1/(2*Nd*mu*ts**2),
}

# --- SeisCL grad_coefvisc_1(), calc_grad.c ---------------------------------
f1 = (Nd*M*(1+L*tp)*(1+a*ts) - 2*(Nd-1)*mu*(1+L*ts)*(1+a*tp))**2
f2 = (Nd*M*tp*(1+a*ts)       - 2*(Nd-1)*mu*ts*(1+a*tp))**2
seiscl = {
 'c1M':  (1+L*tp)*(1+a*tp)*(1+a*ts)**2/f1,
 'c2M':  tp*(1+a*tp)*(1+a*ts)**2/f2,
 'c1tp': M*(L-a)*(1+a*ts)**2/f1,          'c2tp': M*(1+a*ts)**2/f2,
 'c1mu': (1+a*ts)/(mu**2*(1+L*ts)),       'c1ts': (L-a)/(mu*(1+L*ts)**2),
 'c2mu': (Nd+1)/3*(1+L*ts)*(1+a*ts)*(1+a*tp)**2/f1,
 'c2ts': (Nd+1)/3*mu*(L-a)*(1+a*tp)**2/f1,
 'c3mu': (1+a*ts)/(2*Nd*mu**2*(1+L*ts)),  'c3ts': (L-a)/(2*Nd*mu*(1+L*ts)**2),
 'c4mu': (1+a*ts)/(mu**2*ts),             'c4ts': 1/(mu*ts**2),
 'c5mu': (Nd+1)/3*ts*(1+a*ts)*(1+a*tp)**2/f2,
 'c5ts': (Nd+1)/3*mu*(1+a*tp)**2/f2,
 'c6mu': (1+a*ts)/(2*Nd*mu**2*ts),        'c6ts': 1/(2*Nd*mu*ts**2),
}

print("%-6s %-22s %s" % ("coef", "general (any a, L)", "as SeisCL runs (a=0, L=1)"))
for k in thesis:
    gen = sp.simplify(thesis[k] - seiscl[k]) == 0
    run = sp.simplify((thesis[k] - seiscl[k]).subs({a: 0, L: 1})) == 0
    print("%-6s %-22s %s" % (k, "match" if gen else "differ by an L factor",
                             "MATCH" if run else "DIFFER"))

coef   general (any a, L)     as SeisCL runs (a=0, L=1)


c1M    match                  MATCH
c2M    match                  MATCH


c1tp   differ by an L factor  MATCH
c2tp   match                  MATCH
c1mu   match                  MATCH
c1ts   differ by an L factor  MATCH
c2mu   match                  MATCH


c2ts   differ by an L factor  MATCH
c3mu   match                  MATCH
c3ts   differ by an L factor  MATCH
c4mu   match                  MATCH
c4ts   match                  MATCH
c5mu   match                  MATCH
c5ts   match                  MATCH
c6mu   match                  MATCH
c6ts   match                  MATCH


**Result.** Twelve of the sixteen coefficients match identically for
arbitrary $\alpha$ and $L$. The four that do not — $c_1^{\tau_p}$,
$c_1^{\tau_s}$, $c_2^{\tau_s}$, $c_3^{\tau_s}$ — are exactly the ones carrying
$(L-\alpha)$ in the code against $(1-\alpha)$ in the thesis, which is the
per-mechanism vs total $\tau$ convention noted above. **In the regime SeisCL
actually runs ($\alpha=0$), all sixteen agree.**

A caution for anyone re-deriving from a text extraction of the thesis: the
OCR'd text is internally inconsistent about where $L$ appears (it renders
$c_1^{\tau_s}$ with $(1+\tau_s)^2$ but $c_3^{\tau_s}$ with $(1+L\tau_s)^2$).
Check the PDF visually before concluding anything from the text dump.

## 5. From the time domain to the DFT gradient

`back_prop_type=2` evaluates these correlations in the **frequency domain**,
over a chosen set of frequencies, instead of accumulating them per time step.
With Parseval and $\partial_t \rightarrow i\omega$, the stress terms map
cleanly. For $P_1$:

$$\langle\tilde\sigma, \partial_t\sigma\rangle \;=\; \int \omega\,\mathrm{Im}\!\big(\tilde\sigma\,\overline{\sigma}\big)\,d\omega$$

which is exactly what the code computes:

```c
cl_itreal(a,b) = a.y*b.x - a.x*b.y      /* = Im(a * conj(b)) */
dot[0] = freq * cl_itreal(sxxzzr, sxxzz) / dftnorm;
```

Let's confirm that identity numerically.

In [2]:
import numpy as np

rng = np.random.default_rng(0)
A = rng.standard_normal(6) + 1j*rng.standard_normal(6)   # adjoint spectrum
B = rng.standard_normal(6) + 1j*rng.standard_normal(6)   # forward spectrum
w = 2*np.pi*17.0

def cl_itreal(a, b):
    """calc_grad.c:31 -- a.y*b.x - a.x*b.y."""
    return a.imag*b.real - a.real*b.imag

lhs = w * cl_itreal(A, B)                 # what the code forms
rhs = np.real(np.conj(A) * (1j*w) * B)    # <adj, d/dt fwd> via Parseval
print("max|code - theory| =", np.abs(lhs - rhs).max())

max|code - theory| = 5.684341886080802e-14


So the **stress** half of the DFT gradient is a faithful frequency-domain
rendering of eq. (3.53a,c,d).

## 6. What is still open

The **memory-variable** terms $P_2$, $P_5$, $P_6$ carry the operator
$(1+\tau_\sigma^l\partial_t)$. In the code that is `cl_rm`:

```c
cl_rm(a,b,tausig,w) = tausig*(a.x*b.x + a.y*b.y) + (a.x*b.y - a.y*b.x)/w
                    = tausig*Re(a·conj(b)) - Im(a·conj(b))/w
```

Applying Parseval to the thesis form instead gives

$$P_2 = \mathrm{Re}\big(\tilde R\,\overline R\big) + \omega\,\tau_\sigma\,\mathrm{Im}\big(\tilde R\,\overline R\big)$$

These are **not** the same expression, and not proportional — the code puts
$\tau_\sigma$ on the real part and $-1/\omega$ on the imaginary part, the
theory puts $1$ and $+\omega\tau_\sigma$. Let's make the discrepancy
concrete rather than take it on trust.

In [3]:
tausig = 1.0/(2*np.pi*30.0)

def cl_rm(a, b, tausig, w):
    """calc_grad.c:72."""
    return tausig*(a.real*b.real + a.imag*b.imag) \
         + (a.real*b.imag - a.imag*b.real)/w

code_form   = cl_rm(A, B, tausig, w)
theory_form = np.real(np.conj(A)*(1 + 1j*w*tausig)*B)

print("code   :", np.round(code_form[:3], 6))
print("theory :", np.round(theory_form[:3], 6))
ratio = code_form/theory_form
print("ratio  :", np.round(ratio[:3], 6), " (constant? ->",
      bool(np.allclose(ratio, ratio[0])), ")")

code   : [ 0.030166  0.006042 -0.012732]
theory : [-1.502931  0.976869 -0.163961]
ratio  : [-0.020071  0.006185  0.077655]  (constant? -> False )


The ratio is not constant, so this is a structural difference rather
than a convention factor that would cancel out of a gradient direction.

**Why it matters.** The $\tau$ gradients weight the memory terms far more
heavily than the $\mu$ gradient does — for a typical model
$c_4^{\tau_s}/c_4^{\mu} \approx 2\times10^{12}$ against
$c_2^{\tau_s}/c_2^{\mu} \approx 4\times10^{9}$, a factor ~500. So an error
confined to $P_2$/$P_5$/$P_6$ would leave $\partial X/\partial M$,
$\partial X/\partial\mu$ and $\partial X/\partial\rho$ looking right while
corrupting $\partial X/\partial\tau_p$ and $\partial X/\partial\tau_s$ — which
is exactly the behaviour currently observed in SeisCL.

**This is not yet a confirmed bug.** The most likely benign explanation is
that SeisCL's memory variables $r$ differ from the thesis's by a factor — a
$1/(i\omega)$ time integration, or a $1/\tau_\sigma$ scaling in how
`update_s*.cl` defines the recursion — which would absorb the difference.
Settling it means checking the memory-variable update in `update_s2D.cl`
against thesis eq. (3.13). Until then:

> **Treat `gradtaup` and `gradtaus` as unvalidated.** The $(\rho, M, \mu)$
> gradients are on much firmer ground: they are dominated by the stress
> correlations verified in §5, and they match an independent float64
> reference cell by cell.

## Summary

| | status |
|---|---|
| gradient structure (eq. 3.52) vs `calc_grad.c` accumulation | matches term for term |
| $P_i \leftrightarrow$ `dot[]` mapping | established, incl. the `dot[3]=dot[0]`, `dot[6]=dot[1]` reuse |
| coefficients (eq. 3.54–3.55) vs `grad_coefvisc_1` | **all 16 match** at $\alpha=0$; 12 match for arbitrary $\alpha,L$ |
| stress dot products in the DFT gradient | verified against Parseval |
| memory-variable dot products (`cl_rm`) | **open** — does not reduce to $(1+\tau_\sigma\partial_t)$ as written |

Further reading: `notes/viscoelastic-gradient-theory.md` (the maintainer-facing
version of this, with the code line numbers) and
`notes/adjoint-theory-thesis.md` (the adjoint state itself — why the forward
kernels can be reused time-reversed).